In [ ]:
import pandas as pd
import numpy as np

# =================================================================================
# 0. SETTING UTAMA
# =================================================================================

file_asli = "/Users/gitajilanmahira/Downloads/Gita Jilan_Rifaskes 2/01_Rifaskes_PKM.xlsx"
file_dinkes = "/Users/gitajilanmahira/Downloads/Gita Jilan_Rifaskes 2/01_Rifaskes_Dinkes.xlsx"
file_simpan = "/Users/gitajilanmahira/Downloads/Gita Jilan_Rifaskes 2/Siapbab4.xlsx"

# Variabel ini tetap boleh dipakai saat cleaning,
# tetapi akan DIHAPUS dari df_bersih sebelum Excel final disimpan.
VARIABEL_HAPUS_MODEL = [
    "B4R14",     # Total kunjungan pasien rawat jalan
    "B3R11E",    # Jumlah penduduk wilayah kerja
    "B9R13A",    # Jumlah komputer berfungsi
    "B4R16A",    # Total kunjungan pasien rawat inap
    "B3R13A"     # Jumlah Puskesmas pembantu
]

print("Variabel yang akan dihapus dari model final:")
print(VARIABEL_HAPUS_MODEL)

# =================================================================================
# 1. BACA DATA
# =================================================================================

print("\nMembaca data RIFASKES PKM...")
df = pd.read_excel(file_asli)
df.columns = df.columns.str.strip()

# TRACKER 1: JUMLAH AWAL
total_pkm_awal = len(df)

# =================================================================================
# 2. SELEKSI KOLOM VIP (WHITELIST)
# =================================================================================

kolom_dasar = ['ID_cases', 'B1R1', 'B1R2']

variabel_y = [
    'B1281R4K3', 'B1281R265K3', 'B1281R5K3', 'B1281R12K3', 'B1281R165K3',
    'B1281R13K3', 'B1281R14K3', 'B1281R22K3', 'B1281R11K3', 'B1281R41K3',
    'B1281R51K3', 'B1281R58K3', 'B1281R63K3', 'B1281R65K3', 'B1281R2K3',
    'B1281R361K3', 'B1281R115K3', 'B1281R132K3', 'B1281R117K3',
    'B1281R121K3', 'B1281R215K3', 'B1281R193K3', 'B1281R171K3',
    'B1281R185K3', 'B1281R202K3', 'B1281R211K3', 'B1281R217K3',
    'B1281R233K3', 'B1281R243K3', 'B1281R250K3', 'B1281R256K3',
    'B1281R257K3', 'B1281R297K3', 'B1281R183K3', 'B1281R184K3',
    'B1281R248K3', 'B1281R312K3', 'B1281R322K3', 'B1281R359K3',
    'B1281R367K3'
]

variabel_x = [
    'B3R3', 'B3R10', 'B3R9B', 'B3R6', 'B3R11E', 'B3R13A', 'B14R2',
    'B4R14', 'B4R16A', 'B14R23', 'B14R15BTERSEDIA', 'B1282R6',
    'B8R2K3N', 'B8R2K4N', 'B4R10B', 'B121R1', 'B121R3A', 'B121R3B',
    'B121R4', 'B121R5', 'B121R7', 'B121R10', 'B122R3', 'B122R4A',
    'B122R4B', 'B122R5', 'B122R6', 'B13R2ATERSEDIA',
    'B13R2BTERSEDIA', 'B13R2CTERSEDIA', 'B13R2DTERSEDIA',
    'B13R2ETERSEDIA', 'B1282R1', 'B1282R2', 'B1282R3', 'B1282R4',
    'B1282R5', 'B6R1LK2', 'B6R1LK4', 'B122R11', 'B122R11A',
    'B122R11B', 'B122R11C', 'B122R11E', 'B122R11F', 'B122R11G',
    'B122R1', 'B14R13B', 'B4R3', 'B51R1B', 'B3R22E',
    'B4R6A1K2', 'B4R6A2K2', 'B4R6A3K2', 'B3R24', 'B3R40',
    'B51R2DK2', 'B9R13A', 'B125R2', 'B125R1', 'B14R16',
    'B14R7A', 'B126R4A', 'B9R7AK2', 'B9R7BK2', 'B9R7CK2',
    'B9R7DK2', 'B9R7HK2', 'B122R7A', 'B122R7B', 'B122R7C',
    'B122R7D'
]

kolom_skip_logic = ['B3R8', 'B3R5A', 'B51R1A', 'B14R1A', 'B3R23A']

# Gabungkan dan hapus duplikat, tetapi tetap jaga urutan
kolom_vip = list(dict.fromkeys(kolom_dasar + variabel_y + variabel_x + kolom_skip_logic))

print("\nMenyapu bersih kolom yang tidak penting...")
kolom_vip_yg_ada = [col for col in kolom_vip if col in df.columns]
df = df[kolom_vip_yg_ada]
print(f"Beres! Data sisa {len(df.columns)} kolom penting.")

# =================================================================================
# 3. EKSEKUSI SKIP LOGIC & IMPUTASI
# =================================================================================

def aman_loc(col_kondisi, val_kondisi, col_target, val_target):
    if col_kondisi in df.columns and col_target in df.columns:
        mask = (df[col_kondisi] == val_kondisi) & (df[col_target].isna())
        df.loc[mask, col_target] = val_target

print("\n--- Eksekusi Skip Logic ---")

aman_loc('B3R8', 3, 'B3R9B', 0)
aman_loc('B3R5A', 3, 'B3R6', 0)
aman_loc('B14R1A', 2, 'B14R2', 0)
aman_loc('B3R3', 2, 'B4R16A', 0)
aman_loc('B121R1', 2, 'B121R3A', 0)
aman_loc('B121R1', 2, 'B121R3B', 0)
aman_loc('B121R1', 2, 'B121R4', 0)
aman_loc('B121R1', 2, 'B121R5', 0)
aman_loc('B121R7', 1, 'B121R10', 2)
aman_loc('B122R3', 4, 'B122R4A', 0)
aman_loc('B122R3', 4, 'B122R4B', 0)
aman_loc('B122R3', 4, 'B122R6', 0)
aman_loc('B1282R1', 2, 'B1282R2', 0)
aman_loc('B1282R3', 2, 'B1282R4', 0)
aman_loc('B6R1LK2', 3, 'B6R1LK4', 0)
aman_loc('B122R11', 2, 'B122R11A', 0)
aman_loc('B122R11', 2, 'B122R11B', 0)
aman_loc('B122R11', 2, 'B122R11C', 0)
aman_loc('B122R11', 2, 'B122R11E', 0)
aman_loc('B122R11', 2, 'B122R11F', 0)
aman_loc('B122R11', 2, 'B122R11G', 0)
aman_loc('B14R1A', 2, 'B14R13B', 0)
aman_loc('B14R1A', 2, 'B14R16', 0)
aman_loc('B14R1A', 2, 'B14R7A', 0)
aman_loc('B122R3', 4, 'B122R7A', 0)
aman_loc('B122R3', 4, 'B122R7B', 0)
aman_loc('B122R3', 4, 'B122R7C', 0)
aman_loc('B122R3', 4, 'B122R7D', 0)
aman_loc('B51R1A', 3, 'B51R1B', 0)
aman_loc('B3R23A', 2, 'B3R24', 0)

print(">> Mengamankan E-Purchasing (diisi dengan 0)")
if 'B122R5' in df.columns:
    df['B122R5'] = df['B122R5'].fillna(0)

if 'B122R6' in df.columns:
    df['B122R6'] = df['B122R6'].fillna(0)

nan_sekarang = df.isna().sum()
kolom_nan_minor = nan_sekarang[nan_sekarang > 0].index.tolist()

pkm_sebelum_drop_nan = len(df)
df = df.dropna(subset=kolom_nan_minor)
pkm_gugur_nan_minor = pkm_sebelum_drop_nan - len(df)

# =================================================================================
# 4. HAPUS BARIS PERMANEN
# =================================================================================

print("\nMenghapus baris siluman fatal...")

aturan_hapus_baris = {
    'B4R14': [888888, 777777, 999997],
    'B4R16A': [777777, 888888, 999997],
    'B14R15BTERSEDIA': [0, 8, 9, 5],
    'B8R2K3N': [88],
    'B121R4': [8],
    'B1282R2': [888, 88, 77],
    'B1282R4': [-4, 88, 77, 888, 96],
    'B9R13A': [88, 99, 97],
    'B14R7A': [3, 7, 8, 9],
    'B9R7DK2': [88]
}

pkm_sebelum_fatal = len(df)

for kolom, angka_fatal in aturan_hapus_baris.items():
    if kolom in df.columns:
        df = df[~df[kolom].isin(angka_fatal)]

pkm_gugur_fatal = pkm_sebelum_fatal - len(df)

print("Mengubah siluman B14R23 jadi -1...")
if 'B14R23' in df.columns:
    df['B14R23'] = df['B14R23'].replace([88888, 77777, 99999, 99997], -1)

# =================================================================================
# 5. FEATURE ENGINEERING
# =================================================================================

print("\nMemulai proses peracikan variabel baru...")

# --- B. PENGIRIMAN RKO ---
if 'B121R3A' in df.columns and 'B121R3B' in df.columns:
    kondisi_rko = [
        (df['B121R3A'] == 1) & (df['B121R3B'] == 1),
        (df['B121R3A'] == 1) & (df['B121R3B'].isin([2, 0])),
        (df['B121R3A'].isin([2, 0])) & (df['B121R3B'] == 1),
        (df['B121R3A'].isin([2, 0])) & (df['B121R3B'].isin([2, 0]))
    ]
    df['X_Pengiriman_RKO'] = np.select(kondisi_rko, [3, 1, 2, 0], default=np.nan)

# --- E. METODE PEMBELIAN OBAT KAPITASI ---
if 'B122R4A' in df.columns and 'B122R4B' in df.columns:
    kondisi_metode = [
        (df['B122R4A'] == 1) & (df['B122R4B'] == 1),
        ((df['B122R4A'] == 1) & df['B122R4B'].isin([2, 0])) |
        (df['B122R4A'].isin([2, 0]) & (df['B122R4B'] == 1)),
        df['B122R4A'].isin([2, 0]) & df['B122R4B'].isin([2, 0]),
        df['B122R4A'].isin([8]) | df['B122R4B'].isin([8])
    ]
    df['X_Metode_Pembelian_Kapitasi'] = np.select(kondisi_metode, [2, 1, 0, -1], default=np.nan)

# --- PEMENUHAN E-PURCHASING ---
if 'B122R5' in df.columns and 'B122R6' in df.columns:
    kondisi_pemenuhan = [
        (df['B122R5'] == 1),
        (df['B122R5'] == 2) & (df['B122R6'] == 4),
        (df['B122R5'] == 2) & (df['B122R6'] == 3),
        (df['B122R5'] == 2) & (df['B122R6'] == 2),
        (df['B122R5'] == 2) & (df['B122R6'] == 1),
        (df['B122R5'] == 0)
    ]
    df['X_Persentase_Pemenuhan_EPurchasing'] = np.select(
        kondisi_pemenuhan,
        [5, 4, 3, 2, 1, 0],
        default=0
    )
    df = df.drop(columns=['B122R5', 'B122R6'])

# --- C. DANA UMUM ---
kolom_dana_umum = ['B13R2ATERSEDIA', 'B13R2BTERSEDIA', 'B13R2CTERSEDIA']
if set(kolom_dana_umum).issubset(df.columns):
    df['Total_Dana_Umum'] = (
        (df['B13R2ATERSEDIA'] == 1).astype(int) +
        (df['B13R2BTERSEDIA'] == 1).astype(int) +
        (df['B13R2CTERSEDIA'] == 1).astype(int)
    )
    kondisi_dana_umum = [
        (df['Total_Dana_Umum'] == 3),
        (df['Total_Dana_Umum'] == 2),
        (df['Total_Dana_Umum'] == 1),
        (df['Total_Dana_Umum'] == 0)
    ]
    df['X_Sumber_Dana_Umum'] = np.select(kondisi_dana_umum, [3, 2, 1, 0], default=np.nan)
    df = df.drop(columns=['Total_Dana_Umum'])

# --- F. RUANGAN FARMASI ---
if 'B6R1LK2' in df.columns and 'B6R1LK4' in df.columns:
    kondisi_ruangan = [
        (df['B6R1LK2'] == 1) & (df['B6R1LK4'] == 1),
        (df['B6R1LK2'] == 1) & (df['B6R1LK4'] == 2),
        (df['B6R1LK2'] == 1) & (df['B6R1LK4'] == 3),
        (df['B6R1LK2'] == 2) & (df['B6R1LK4'] == 1),
        (df['B6R1LK2'] == 2) & (df['B6R1LK4'] == 2),
        (df['B6R1LK2'] == 2) & (df['B6R1LK4'] == 3),
        (df['B6R1LK2'] == 3)
    ]
    df['X_Kondisi_Ruang_Farmasi'] = np.select(kondisi_ruangan, [6, 5, 4, 3, 2, 1, 0], default=np.nan)

# --- G. KUALITAS GUDANG OBAT ---
kolom_gudang = ['B122R11A', 'B122R11B', 'B122R11C', 'B122R11E', 'B122R11F', 'B122R11G']
if set(kolom_gudang).issubset(df.columns):
    is_nan = df['B122R11A'].isna()
    df['Skor_Gudang'] = (
        (df['B122R11A'] == 1).astype(int) +
        (df['B122R11B'] == 1).astype(int) +
        (df['B122R11C'] == 1).astype(int) +
        (df['B122R11E'] == 1).astype(int) +
        (df['B122R11F'] == 1).astype(int) +
        (df['B122R11G'] == 1).astype(int)
    )
    kondisi_gudang = [
        (df['Skor_Gudang'] == 6) & ~is_nan,
        (df['Skor_Gudang'].isin([4, 5])) & ~is_nan,
        (df['Skor_Gudang'].isin([2, 3])) & ~is_nan,
        (df['Skor_Gudang'].isin([0, 1])) & ~is_nan,
        is_nan
    ]
    df['X_Kualitas_Gudang'] = np.select(kondisi_gudang, [4, 3, 2, 1, 0], default=np.nan)
    df = df.drop(columns=kolom_gudang + ['Skor_Gudang'])

# --- A. DANA BPJS ---
if 'B13R2DTERSEDIA' in df.columns and 'B13R2ETERSEDIA' in df.columns:
    kondisi_bpjs = [
        (df['B13R2DTERSEDIA'] == 1) & (df['B13R2ETERSEDIA'] == 1),
        (df['B13R2DTERSEDIA'] == 1) & (df['B13R2ETERSEDIA'].isin([2, 0])),
        (df['B13R2DTERSEDIA'].isin([2, 0])) & (df['B13R2ETERSEDIA'] == 1),
        (df['B13R2DTERSEDIA'].isin([2, 0])) & (df['B13R2ETERSEDIA'].isin([2, 0]))
    ]
    df['X_Sumber_Dana_BPJS'] = np.select(kondisi_bpjs, [3, 2, 1, 0], default=np.nan)

# --- H. PROPORSI KAPITASI ---
if 'B14R13B' in df.columns:
    df['B14R13B'] = df['B14R13B'].fillna(0)
    df['X_Proporsi_Kapitasi'] = np.where(df['B14R13B'] <= 40.0, 1, 0)
    df = df.drop(columns=['B14R13B'])

# --- I. SISTEM PENCATATAN ---
kolom_sistem = ['B4R6A1K2', 'B4R6A2K2', 'B4R6A3K2']
if set(kolom_sistem).issubset(df.columns):
    cond_tidak_ada = df[kolom_sistem].isin([4, 0, np.nan]).all(axis=1)
    ada_campuran = (df[kolom_sistem] == 2).any(axis=1)
    ada_elektronik = (df[kolom_sistem] == 1).any(axis=1)
    ada_manual = (df[kolom_sistem] == 3).any(axis=1)

    cond_campuran_final = ada_campuran | (ada_elektronik & ada_manual)
    cond_elektronik_final = ada_elektronik & ~cond_campuran_final & ~ada_manual
    cond_manual_final = ada_manual & ~cond_campuran_final & ~ada_elektronik

    kondisi_sistem = [
        cond_elektronik_final,
        cond_manual_final,
        cond_campuran_final,
        cond_tidak_ada
    ]
    df['X_Sistem_Pencatatan'] = np.select(kondisi_sistem, [3, 2, 1, 0], default=np.nan)

# --- K. KERUSAKAN OBAT ---
if 'B125R1' in df.columns and 'B125R2' in df.columns:
    kondisi_rusak = [
        (df['B125R1'] == 1) & (df['B125R2'] == 1),
        (df['B125R1'] == 1) & (df['B125R2'].isin([2, 0])),
        (df['B125R1'].isin([2, 0])) & (df['B125R2'] == 1),
        (df['B125R1'].isin([2, 0])) & (df['B125R2'].isin([2, 0]))
    ]
    df['X_Kerusakan_Obat'] = np.select(kondisi_rusak, [0, 1, 2, 3], default=np.nan)

# --- L. KEPEMILIKAN KENDARAAN ---
kolom_kendaraan = ['B9R7AK2', 'B9R7BK2', 'B9R7CK2', 'B9R7DK2', 'B9R7HK2']
if set(kolom_kendaraan).issubset(df.columns):
    punya_roda_4 = (
        df['B9R7AK2'].between(1, 87) |
        df['B9R7BK2'].between(1, 87) |
        df['B9R7CK2'].between(1, 87)
    )
    punya_roda_2 = df['B9R7DK2'].between(1, 87)

    kondisi_kendaraan = [
        punya_roda_4 & punya_roda_2,
        punya_roda_4 & ~punya_roda_2,
        ~punya_roda_4 & punya_roda_2,
        ~punya_roda_4 & ~punya_roda_2
    ]
    df['X_Kepemilikan_Kendaraan'] = np.select(kondisi_kendaraan, [3, 2, 1, 0], default=np.nan)

# --- D. INTENSITAS KAPITASI ---
kolom_kapitasi = ['B122R7A', 'B122R7B', 'B122R7C', 'B122R7D']
if set(kolom_kapitasi).issubset(df.columns):
    df['X_Intensitas_Kapitasi'] = (
        (df['B122R7A'] == 1).astype(int) +
        (df['B122R7B'] == 1).astype(int) +
        (df['B122R7C'] == 1).astype(int) +
        (df['B122R7D'] == 1).astype(int)
    )
    df = df.drop(columns=kolom_kapitasi)

# =================================================================================
# 6. FINALISASI: BUANG KOLOM BAHAN BAKU
# =================================================================================

kolom_dibuang = [
    'B13R2DTERSEDIA', 'B13R2ETERSEDIA',
    'B121R3A', 'B121R3B',
    'B13R2ATERSEDIA', 'B13R2BTERSEDIA', 'B13R2CTERSEDIA',
    'B122R4A', 'B122R4B',
    'B6R1LK2', 'B6R1LK4',
    'B122R11A', 'B122R11B', 'B122R11C', 'B122R11E', 'B122R11F', 'B122R11G',
    'B4R6A1K2', 'B4R6A2K2', 'B4R6A3K2',
    'B125R1', 'B125R2',
    'B9R7AK2', 'B9R7BK2', 'B9R7CK2', 'B9R7DK2', 'B9R7HK2'
]

df = df.drop(columns=[c for c in kolom_dibuang if c in df.columns])

# =================================================================================
# 7. BUAT VARIABEL TARGET Y (PUSKESMAS)
# =================================================================================

if 'B1281R4K3' in df.columns and 'B1281R265K3' in df.columns:
    df['Y_Obat_Cacing'] = df[['B1281R4K3', 'B1281R265K3']].min(axis=1)

if 'B1281R12K3' in df.columns and 'B1281R165K3' in df.columns:
    df['Y_Hipertensi'] = df[['B1281R12K3', 'B1281R165K3']].min(axis=1)

if 'B1281R115K3' in df.columns and 'B1281R132K3' in df.columns:
    df['Y_Diuretik'] = df[['B1281R115K3', 'B1281R132K3']].min(axis=1)

if 'B1281R121K3' in df.columns and 'B1281R215K3' in df.columns:
    df['Y_Diabetes'] = df[['B1281R121K3', 'B1281R215K3']].min(axis=1)

kolom_mata = [
    c for c in ['B1281R183K3', 'B1281R184K3', 'B1281R248K3']
    if c in df.columns
]

if kolom_mata:
    df['Y_Mata'] = df[kolom_mata].min(axis=1)

mapping_y_tunggal = {
    'B1281R5K3': 'Y_Alopurinol',
    'B1281R13K3': 'Y_Amoksisilin_Sirup',
    'B1281R14K3': 'Y_Amoksisilin_Tablet',
    'B1281R22K3': 'Y_Antasida',
    'B1281R11K3': 'Y_Amitriptilin',
    'B1281R41K3': 'Y_Asiklovir',
    'B1281R51K3': 'Y_Betametason_Krim',
    'B1281R58K3': 'Y_Deksametason',
    'B1281R63K3': 'Y_Diazepam_Injeksi',
    'B1281R65K3': 'Y_Diazepam_Tablet',
    'B1281R2K3': 'Y_Adrenalin_Injeksi',
    'B1281R361K3': 'Y_VitK_Injeksi',
    'B1281R117K3': 'Y_Oralit',
    'B1281R193K3': 'Y_Kotrimoksazol',
    'B1281R171K3': 'Y_Ketokonazol',
    'B1281R185K3': 'Y_CTM',
    'B1281R202K3': 'Y_Lidokain_Injeksi',
    'B1281R211K3': 'Y_MgSO4_Injeksi',
    'B1281R217K3': 'Y_Metilergometrin_Injeksi',
    'B1281R233K3': 'Y_Natrium_Diklofenak',
    'B1281R243K3': 'Y_OAT_Dewasa',
    'B1281R250K3': 'Y_Oksitosin_Injeksi',
    'B1281R256K3': 'Y_Parasetamol_Sirup',
    'B1281R257K3': 'Y_Parasetamol_Tablet',
    'B1281R297K3': 'Y_Salbutamol',
    'B1281R312K3': 'Y_Simvastatin',
    'B1281R322K3': 'Y_TTD_Besi',
    'B1281R359K3': 'Y_VitB6',
    'B1281R367K3': 'Y_Zink'
}

for col_asli, col_baru in mapping_y_tunggal.items():
    if col_asli in df.columns:
        df[col_baru] = df[col_asli]

target_y_potensial = [
    'Y_Obat_Cacing',
    'Y_Hipertensi',
    'Y_Diuretik',
    'Y_Diabetes',
    'Y_Mata'
] + list(mapping_y_tunggal.values())

kolom_y_yg_sukses_dibuat = [
    col for col in target_y_potensial
    if col in df.columns
]

pkm_sebelum_y = len(df)
df = df.dropna(subset=kolom_y_yg_sukses_dibuat)
pkm_gugur_y = pkm_sebelum_y - len(df)

for col in kolom_y_yg_sukses_dibuat:
    df[col] = df[col].replace(2, 0)

df['Total_Kelompok_Obat_PKM'] = df[kolom_y_yg_sukses_dibuat].sum(axis=1)
df['Y_Final_PKM'] = np.where(df['Total_Kelompok_Obat_PKM'] >= 32, 1, 0)

# =================================================================================
# 8. BACA DATA DINKES & MERGE SEBAGAI VARIABEL X NUMERIK
# =================================================================================

print("\nMemproses Data Dinkes dan Merging...")

df_dinkes = pd.read_excel(file_dinkes)
df_dinkes.columns = df_dinkes.columns.str.strip()

def kode_to_str(x):
    if pd.isna(x):
        return np.nan
    try:
        return str(int(float(x)))
    except:
        return str(x).strip()

if 'B66R4K3' in df_dinkes.columns and 'B66R265K3' in df_dinkes.columns:
    df_dinkes['X_Obat_Cacing'] = df_dinkes[['B66R4K3', 'B66R265K3']].min(axis=1)

if 'B66R12K3' in df_dinkes.columns and 'B66R165K3' in df_dinkes.columns:
    df_dinkes['X_Hipertensi'] = df_dinkes[['B66R12K3', 'B66R165K3']].min(axis=1)

if 'B66R115K3' in df_dinkes.columns and 'B66R132K3' in df_dinkes.columns:
    df_dinkes['X_Diuretik'] = df_dinkes[['B66R115K3', 'B66R132K3']].min(axis=1)

if 'B66R121K3' in df_dinkes.columns and 'B66R215K3' in df_dinkes.columns:
    df_dinkes['X_Diabetes'] = df_dinkes[['B66R121K3', 'B66R215K3']].min(axis=1)

kolom_mata_d = [
    c for c in ['B66R183K3', 'B66R184K3', 'B66R248K3']
    if c in df_dinkes.columns
]

if kolom_mata_d:
    df_dinkes['X_Mata'] = df_dinkes[kolom_mata_d].min(axis=1)

mapping_x_tunggal = {
    'B66R5K3': 'X_Alopurinol',
    'B66R13K3': 'X_Amoksisilin_Sirup',
    'B66R14K3': 'X_Amoksisilin_Tablet',
    'B66R22K3': 'X_Antasida',
    'B66R11K3': 'X_Amitriptilin',
    'B66R41K3': 'X_Asiklovir',
    'B66R51K3': 'X_Betametason_Krim',
    'B66R58K3': 'X_Deksametason',
    'B66R63K3': 'X_Diazepam_Injeksi',
    'B66R65K3': 'X_Diazepam_Tablet',
    'B66R2K3': 'X_Adrenalin_Injeksi',
    'B66R361K3': 'X_VitK_Injeksi',
    'B66R117K3': 'X_Oralit',
    'B66R193K3': 'X_Kotrimoksazol',
    'B66R171K3': 'X_Ketokonazol',
    'B66R185K3': 'X_CTM',
    'B66R202K3': 'X_Lidokain_Injeksi',
    'B66R211K3': 'X_MgSO4_Injeksi',
    'B66R217K3': 'X_Metilergometrin_Injeksi',
    'B66R233K3': 'X_Natrium_Diklofenak',
    'B66R243K3': 'X_OAT_Dewasa',
    'B66R250K3': 'X_Oksitosin_Injeksi',
    'B66R256K3': 'X_Parasetamol_Sirup',
    'B66R257K3': 'X_Parasetamol_Tablet',
    'B66R297K3': 'X_Salbutamol',
    'B66R312K3': 'X_Simvastatin',
    'B66R322K3': 'X_TTD_Besi',
    'B66R359K3': 'X_VitB6',
    'B66R367K3': 'X_Zink'
}

for col_asli, col_baru in mapping_x_tunggal.items():
    if col_asli in df_dinkes.columns:
        df_dinkes[col_baru] = df_dinkes[col_asli]

target_x_potensial = [
    'X_Obat_Cacing',
    'X_Hipertensi',
    'X_Diuretik',
    'X_Diabetes',
    'X_Mata'
] + list(mapping_x_tunggal.values())

kolom_x_sukses = [
    col for col in target_x_potensial
    if col in df_dinkes.columns
]

print("Jumlah kelompok obat Dinkes yang berhasil dibuat:", len(kolom_x_sukses))

# 1 = tersedia, selain 1 = tidak tersedia
for col in kolom_x_sukses:
    df_dinkes[col] = np.where(df_dinkes[col] == 1, 1, 0)

df_dinkes['Total_Kelompok_Obat_Dinkes'] = df_dinkes[kolom_x_sukses].sum(axis=1)

print("\nRingkasan Total_Kelompok_Obat_Dinkes:")
print(df_dinkes['Total_Kelompok_Obat_Dinkes'].describe())

# Merge berdasarkan provinsi & kabupaten
df['B1R1_str'] = df['B1R1'].apply(kode_to_str)
df['B1R2_str'] = df['B1R2'].apply(kode_to_str).str.zfill(2)
df['KODE_KAB_PUSKESMAS'] = df['B1R1_str'] + df['B1R2_str']

df_dinkes['propkab'] = df_dinkes['propkab'].apply(kode_to_str)

df_dinkes_subset = df_dinkes[
    [
        'propkab',
        'Total_Kelompok_Obat_Dinkes'
    ]
].copy()

df_dinkes_subset = (
    df_dinkes_subset
    .dropna(subset=['propkab'])
    .groupby('propkab', as_index=False)['Total_Kelompok_Obat_Dinkes']
    .mean()
)

df = pd.merge(
    df,
    df_dinkes_subset,
    left_on='KODE_KAB_PUSKESMAS',
    right_on='propkab',
    how='left'
)

print("\nCek missing Total_Kelompok_Obat_Dinkes setelah merge:")
missing_dinkes = df['Total_Kelompok_Obat_Dinkes'].isna().sum()
print(missing_dinkes)

if missing_dinkes > 0:
    median_dinkes = df['Total_Kelompok_Obat_Dinkes'].median()
    df['Total_Kelompok_Obat_Dinkes'] = df['Total_Kelompok_Obat_Dinkes'].fillna(median_dinkes)
    print("Missing Dinkes diisi median:", median_dinkes)

df = df.drop(
    columns=[
        'B1R1_str',
        'B1R2_str',
        'KODE_KAB_PUSKESMAS',
        'propkab'
    ]
)

# =================================================================================
# 9. SAPU BERSIH KOLOM BAHAN BAKU & BOCORAN ANTI-NYONTEK
# =================================================================================

df_bersih = df.copy()

# Buang bahan mentah obat PKM
kolom_obat_mentah_pkm = variabel_y
df_bersih = df_bersih.drop(columns=[c for c in kolom_obat_mentah_pkm if c in df_bersih.columns])

# Buang Y perantara, sisakan Y_Final_PKM
kolom_bocoran = [
    col for col in df_bersih.columns
    if col.startswith('Y_') and col != 'Y_Final_PKM'
]

df_bersih = df_bersih.drop(columns=[c for c in kolom_bocoran if c in df_bersih.columns])

# Buang kolom syarat skip logic
sampah_sisa = ['B51R1A', 'B3R8', 'B3R5A', 'B14R1A', 'B3R23A']
df_bersih = df_bersih.drop(columns=[c for c in sampah_sisa if c in df_bersih.columns])

# =================================================================================
# 10. MAPPING NILAI ORDINAL
# =================================================================================

print("\nMelakukan mapping variabel...")

if 'B3R6' in df_bersih.columns:
    df_bersih['B3R6'] = df_bersih['B3R6'].replace({3: 1, 1: 3})

if 'B3R3' in df_bersih.columns:
    df_bersih['B3R3'] = df_bersih['B3R3'].replace({2: 0})

if 'B3R10' in df_bersih.columns:
    df_bersih['B3R10'] = df_bersih['B3R10'].replace({3: 0, 2: 1, 1: 2})

if 'B14R2' in df_bersih.columns:
    df_bersih['B14R2'] = df_bersih['B14R2'].replace({2: 1, 1: 2})

if 'B1282R6' in df_bersih.columns:
    df_bersih['B1282R6'] = df_bersih['B1282R6'].replace({2: 0})

if 'B8R2K4N' in df_bersih.columns:
    df_bersih['B8R2K4N'] = df_bersih['B8R2K4N'].replace({3: 0, 2: 1, 1: 2})

if 'B4R10B' in df_bersih.columns:
    df_bersih['B4R10B'] = df_bersih['B4R10B'].replace({3: 0, 2: 1, 1: 2})

if 'B121R1' in df_bersih.columns:
    df_bersih['B121R1'] = df_bersih['B121R1'].replace({2: 0})

if 'B121R5' in df_bersih.columns:
    df_bersih['B121R5'] = df_bersih['B121R5'].replace({1: 2, 2: 1})

if 'B121R7' in df_bersih.columns:
    df_bersih['B121R7'] = df_bersih['B121R7'].replace({2: 0})

if 'B121R10' in df_bersih.columns:
    df_bersih['B121R10'] = df_bersih['B121R10'].replace({2: 0})

if 'B122R3' in df_bersih.columns:
    df_bersih['B122R3'] = df_bersih['B122R3'].replace({4: 0})

if 'B1282R1' in df_bersih.columns:
    df_bersih['B1282R1'] = df_bersih['B1282R1'].replace({2: 0})

if 'B1282R3' in df_bersih.columns:
    df_bersih['B1282R3'] = df_bersih['B1282R3'].replace({2: 0})

if 'B122R11' in df_bersih.columns:
    df_bersih['B122R11'] = df_bersih['B122R11'].replace({2: 0})

if 'B122R1' in df_bersih.columns:
    df_bersih['B122R1'] = df_bersih['B122R1'].replace({3: 0})

if 'B4R3' in df_bersih.columns:
    df_bersih['B4R3'] = df_bersih['B4R3'].replace({3: 0, 2: 1, 1: 2})

if 'B51R1B' in df_bersih.columns:
    df_bersih['B51R1B'] = df_bersih['B51R1B'].replace({6: 1, 5: 2, 4: 3, 3: 4, 2: 5})

if 'B51R2DK2' in df_bersih.columns:
    df_bersih['B51R2DK2'] = df_bersih['B51R2DK2'].replace({2: 0})

if 'B3R22E' in df_bersih.columns:
    df_bersih['B3R22E'] = df_bersih['B3R22E'].replace({2: 0})

if 'B3R24' in df_bersih.columns:
    df_bersih['B3R24'] = df_bersih['B3R24'].replace({4: 1, 3: 2, 2: 3, 1: 4})

if 'B3R40' in df_bersih.columns:
    df_bersih['B3R40'] = df_bersih['B3R40'].replace({2: 0})

if 'B1282R5' in df_bersih.columns:
    df_bersih['B1282R5'] = df_bersih['B1282R5'].replace({2: 0})

if 'B14R16' in df_bersih.columns:
    df_bersih['B14R16'] = df_bersih['B14R16'].replace({2: 1, 1: 2})

if 'B126R4A' in df_bersih.columns:
    df_bersih['B126R4A'] = df_bersih['B126R4A'].replace({2: 0})

if 'B14R15BTERSEDIA' in df_bersih.columns:
    df_bersih['B14R15BTERSEDIA'] = df_bersih['B14R15BTERSEDIA'].replace({2: 0})

# =================================================================================
# 11. HAPUS VARIABEL YANG TIDAK MASUK MODEL FINAL
# =================================================================================

print("\nMenghapus variabel yang tidak akan dipakai dalam model final...")

kolom_hapus_final = VARIABEL_HAPUS_MODEL + [
    'X_Ketersediaan_Dinkes'
]

kolom_hapus_final_ada = [
    col for col in kolom_hapus_final
    if col in df_bersih.columns
]

df_bersih = df_bersih.drop(columns=kolom_hapus_final_ada)

print("Kolom yang berhasil dihapus dari df_bersih:")
print(kolom_hapus_final_ada)

# =================================================================================
# 12. QUALITY CHECK FINAL
# =================================================================================

print("\n==================================================================")
print("LAPORAN TRACKER ELIMINASI PUSKESMAS")
print("==================================================================")
print(f"1. Total Puskesmas Awal          : {total_pkm_awal}")
print(f"2. Gugur karena NaN Minor        : - {pkm_gugur_nan_minor}")
print(f"3. Gugur karena Siluman Fatal    : - {pkm_gugur_fatal}")
print(f"4. Gugur karena Target Y Kosong  : - {pkm_gugur_y}")
print("------------------------------------------------------------------")
print(f"SISA PUSKESMAS BERSIH           : {len(df_bersih)} Puskesmas")
print("==================================================================")

print("\nCek kolom Dinkes final:")
print("Total_Kelompok_Obat_Dinkes ada?", 'Total_Kelompok_Obat_Dinkes' in df_bersih.columns)
print("X_Ketersediaan_Dinkes ada?", 'X_Ketersediaan_Dinkes' in df_bersih.columns)

print("\nCek 5 variabel yang harus hilang:")
for col in VARIABEL_HAPUS_MODEL:
    print(col, "ada?", col in df_bersih.columns)

print("\nDistribusi Y_Final_PKM:")
print(df_bersih['Y_Final_PKM'].value_counts().sort_index())
print((df_bersih['Y_Final_PKM'].value_counts(normalize=True).sort_index() * 100).round(2))

print("\nShape final df_bersih:")
print(df_bersih.shape)

print("\nDaftar kolom final:")
print(df_bersih.columns.tolist())

# =================================================================================
# 13. SIMPAN EXCEL BERSIH
# =================================================================================

df_bersih.to_excel(file_simpan, index=False)
print(f"\nFile berhasil disimpan di: {file_simpan}")

In [ ]:
# ============================================================
# CELL TUNING ULANG DARI FILE Siapbab4.xlsx
# Model:
# 1. RF Baseline sederhana
# 2. RF Baseline + Encoding/Scaling
# 3. RF Tuned
# 4. Threshold Optimization
# ============================================================

import pandas as pd
import numpy as np
import warnings

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

# ============================================================
# 1. BACA DATA FINAL BARU
# ============================================================

file_bersih = "/Users/gitajilanmahira/Downloads/Gita Jilan_Rifaskes 2/Siapbab4.xlsx"

print("Membaca file:", file_bersih)
df_ml = pd.read_excel(file_bersih)
df_ml.columns = df_ml.columns.str.strip()

print("Shape df_ml:", df_ml.shape)

TARGET_COL = "Y_Final_PKM"

if TARGET_COL not in df_ml.columns:
    raise ValueError("Y_Final_PKM tidak ditemukan.")

# ============================================================
# 2. CEK PENGAMAN VARIABEL HAPUS
# ============================================================

VARIABEL_HAPUS_MODEL = [
    "B4R14",
    "B3R11E",
    "B9R13A",
    "B4R16A",
    "B3R13A"
]

print("\nCek 5 variabel yang harus sudah hilang:")
for col in VARIABEL_HAPUS_MODEL:
    print(col, "ada?", col in df_ml.columns)

print("\nCek Dinkes:")
print("Total_Kelompok_Obat_Dinkes ada?", "Total_Kelompok_Obat_Dinkes" in df_ml.columns)
print("X_Ketersediaan_Dinkes ada?", "X_Ketersediaan_Dinkes" in df_ml.columns)

# ============================================================
# 3. BUAT X DAN y
# ============================================================

y = df_ml[TARGET_COL].copy()

daftar_hitam = [
    TARGET_COL,
    "Total_Kelompok_Obat_PKM",
    "ID_cases",
    "B1R1",
    "B1R2",
    "X_Ketersediaan_Dinkes"
]

for col in df_ml.columns:
    if col.startswith("Y_") or "B1281" in col or "B66" in col:
        if col not in daftar_hitam:
            daftar_hitam.append(col)

X = df_ml.drop(columns=[c for c in daftar_hitam if c in df_ml.columns]).copy()

print("\nShape X:", X.shape)
print("Shape y:", y.shape)

print("\nDistribusi y seluruh data:")
print(y.value_counts().sort_index())
print((y.value_counts(normalize=True).sort_index() * 100).round(2))

# ============================================================
# 4. DEFINISI KOLOM
# ============================================================

kolom_passthrough = [
    "B3R3",
    "B1282R6",
    "B14R15BTERSEDIA",
    "B121R1",
    "B121R7",
    "B1282R1",
    "B1282R3",
    "B122R11",
    "B51R2DK2",
    "B3R22E",
    "B1282R5",
    "B3R40",
    "B126R4A",
    "B3R10",
    "B3R9B",
    "B14R2",
    "B8R2K4N",
    "B4R10B",
    "B121R5",
    "B121R10",
    "X_Persentase_Pemenuhan_EPurchasing",
    "X_Sumber_Dana_Umum",
    "X_Kondisi_Ruang_Farmasi",
    "X_Kualitas_Gudang",
    "X_Proporsi_Kapitasi",
    "B4R3",
    "B3R24",
    "B14R16",
    "X_Intensitas_Kapitasi"
]

kolom_nominal = [
    "B3R6",
    "X_Pengiriman_RKO",
    "B121R4",
    "B122R3",
    "X_Metode_Pembelian_Kapitasi",
    "X_Sumber_Dana_BPJS",
    "B122R1",
    "B51R1B",
    "X_Kerusakan_Obat",
    "B14R7A",
    "X_Kepemilikan_Kendaraan",
    "X_Sistem_Pencatatan"
]

kolom_numerik = [
    "B8R2K3N",
    "B1282R2",
    "B1282R4",
    "B14R23",
    "Total_Kelompok_Obat_Dinkes"
]

kolom_passthrough_ada = [c for c in kolom_passthrough if c in X.columns]
kolom_nominal_ada = [c for c in kolom_nominal if c in X.columns]
kolom_numerik_ada = [c for c in kolom_numerik if c in X.columns]

print("\nKolom passthrough:", len(kolom_passthrough_ada), kolom_passthrough_ada)
print("Kolom nominal    :", len(kolom_nominal_ada), kolom_nominal_ada)
print("Kolom numerik    :", len(kolom_numerik_ada), kolom_numerik_ada)

# ============================================================
# 5. SPLIT DATA STRATIFY Y
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nX_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nDistribusi y_train:")
print(y_train.value_counts().sort_index())
print((y_train.value_counts(normalize=True).sort_index() * 100).round(2))

print("\nDistribusi y_test:")
print(y_test.value_counts().sort_index())
print((y_test.value_counts(normalize=True).sort_index() * 100).round(2))

# ============================================================
# 6. FUNGSI EVALUASI
# ============================================================

def evaluasi_model(nama_model, y_true, y_pred, y_proba):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    
    hasil = {
        "model": nama_model,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "auroc": roc_auc_score(y_true, y_proba),
        "auprc": average_precision_score(y_true, y_proba),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }
    
    return hasil

# ============================================================
# 7. MODEL 1: RANDOM FOREST BASELINE SEDERHANA
#    Tanpa encoding/scaling khusus
#    Semua kategori dianggap angka apa adanya
# ============================================================

print("\n" + "="*80)
print("MODEL 1: RANDOM FOREST BASELINE SEDERHANA")
print("="*80)

rf_baseline = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

rf_baseline.fit(X_train, y_train)

y_pred_base = rf_baseline.predict(X_test)
y_proba_base = rf_baseline.predict_proba(X_test)[:, 1]

hasil_base = evaluasi_model(
    "RF Baseline",
    y_test,
    y_pred_base,
    y_proba_base
)

print(hasil_base)

# ============================================================
# 8. MODEL 2: RF BASELINE + ENCODING/SCALING
# ============================================================

print("\n" + "="*80)
print("MODEL 2: RF BASELINE + ENCODING/SCALING")
print("="*80)

preprocessor = ColumnTransformer(
    transformers=[
        ("jalur_nom", OneHotEncoder(handle_unknown="ignore"), kolom_nominal_ada),
        ("jalur_num", StandardScaler(), kolom_numerik_ada)
    ],
    remainder="passthrough"
)

rf_pipeline_baseline = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline_baseline.fit(X_train, y_train)

y_pred_pipe = rf_pipeline_baseline.predict(X_test)
y_proba_pipe = rf_pipeline_baseline.predict_proba(X_test)[:, 1]

hasil_pipe = evaluasi_model(
    "RF Baseline + Encoding/Scaling",
    y_test,
    y_pred_pipe,
    y_proba_pipe
)

print(hasil_pipe)

# ============================================================
# 9. MODEL 3: RANDOM FOREST TUNING
# ============================================================

print("\n" + "="*80)
print("MODEL 3: RANDOM FOREST TUNING")
print("="*80)

rf_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
])

param_dist = {
    "model__n_estimators": [300, 500, 700, 1000],
    "model__max_depth": [10, 20, 30, 40, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", None],
    "model__bootstrap": [True, False]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

tuning_rf = RandomizedSearchCV (
    estimator=rf_pipeline,
    param_distributions=param_dist,
    n_iter=80,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

tuning_rf.fit(X_train, y_train)

best_model_rf = tuning_rf.best_estimator_

print("\nBest Params RF:")
print(tuning_rf.best_params_)

print("\nBest CV AUROC:")
print(tuning_rf.best_score_)

y_pred_tuned_default = best_model_rf.predict(X_test)
y_proba_tuned = best_model_rf.predict_proba(X_test)[:, 1]

hasil_tuned_default = evaluasi_model(
    "RF Tuned Default Threshold 0.50",
    y_test,
    y_pred_tuned_default,
    y_proba_tuned
)

print("\nHasil RF Tuned threshold 0.50:")
print(hasil_tuned_default)

# ============================================================
# 10. THRESHOLD OPTIMIZATION
#     Pakai validation split dari training set
# ============================================================

print("\n" + "="*80)
print("THRESHOLD OPTIMIZATION")
print("="*80)

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# Ambil parameter terbaik dari tuning
model_threshold = tuning_rf.best_estimator_

# Fit ulang pada X_fit untuk cari threshold di validation
model_threshold.fit(X_fit, y_fit)

y_proba_val = model_threshold.predict_proba(X_val)[:, 1]

thresholds = np.arange(0.10, 0.91, 0.01)

scores_mcc = []
scores_f1 = []

for t in thresholds:
    y_pred_val_t = (y_proba_val >= t).astype(int)
    scores_mcc.append(matthews_corrcoef(y_val, y_pred_val_t))
    scores_f1.append(f1_score(y_val, y_pred_val_t, zero_division=0))

best_idx = int(np.argmax(scores_mcc))
best_threshold = float(thresholds[best_idx])
best_mcc_val = float(scores_mcc[best_idx])
best_f1_val = float(scores_f1[best_idx])

print("Best threshold berdasarkan MCC validation:", round(best_threshold, 2))
print("Best MCC validation:", round(best_mcc_val, 4))
print("F1 validation pada threshold terbaik:", round(best_f1_val, 4))

# Fit ulang best model pada seluruh training set
best_model_rf.fit(X_train, y_train)

# Evaluasi test set dengan threshold optimal
y_proba_final = best_model_rf.predict_proba(X_test)[:, 1]
y_pred_final = (y_proba_final >= best_threshold).astype(int)

hasil_final_threshold = evaluasi_model(
    "RF Tuned + Optimized Threshold",
    y_test,
    y_pred_final,
    y_proba_final
)

print("\nHasil final RF Tuned + Threshold Optimal:")
print(hasil_final_threshold)

# ============================================================
# 11. TABEL PERBANDINGAN MODEL
# ============================================================

df_hasil = pd.DataFrame([
    hasil_base,
    hasil_pipe,
    hasil_tuned_default,
    hasil_final_threshold
])

kolom_tampil = [
    "model",
    "accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "mcc",
    "auroc",
    "auprc",
    "tn",
    "fp",
    "fn",
    "tp"
]

df_hasil = df_hasil[kolom_tampil]

print("\n" + "="*80)
print("TABEL PERBANDINGAN MODEL")
print("="*80)
print(df_hasil.round(4).to_string(index=False))

# Simpan hasil pembanding
output_hasil = "/Users/gitajilanmahira/Downloads/Gita Jilan_Rifaskes 2/Hasil_Tuning_Siapbab4.xlsx"
df_hasil.to_excel(output_hasil, index=False)

print("\nFile hasil tuning disimpan di:")
print(output_hasil)

# ============================================================
# 12. FEATURE IMPORTANCE TOP 20 MODEL FINAL
# ============================================================

feature_names = best_model_rf.named_steps["prep"].get_feature_names_out()
importance = best_model_rf.named_steps["model"].feature_importances_

df_feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": importance
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("\n" + "="*80)
print("TOP 20 FEATURE IMPORTANCE MODEL FINAL")
print("="*80)
print(df_feature_importance.head(20).to_string(index=False))

output_fi = "/Users/gitajilanmahira/Downloads/Gita Jilan_Rifaskes 2/Feature_Importance_Siapbab4.xlsx"
df_feature_importance.to_excel(output_fi, index=False)

print("\nFile feature importance disimpan di:")
print(output_fi)

print("\n" + "="*80)
print("RINGKASAN YANG PERLU DIKIRIM KE CHATGPT")
print("="*80)
print("Shape X_train:", X_train.shape)
print("Shape X_test :", X_test.shape)
print("Distribusi y_train:")
print(y_train.value_counts().sort_index())
print("Distribusi y_test:")
print(y_test.value_counts().sort_index())
print("Best Params RF:")
print(tuning_rf.best_params_)
print("Best CV AUROC:", tuning_rf.best_score_)
print("Best Threshold:", best_threshold)
print("Hasil final:")
print(hasil_final_threshold)
print("Top 20 Feature Importance:")
print(df_feature_importance.head(20))

In [ ]:
# ============================================================
# SCRIPT FULL EVALUASI MODEL FINAL TANPA TUNING ULANG
# INPUT  : Siapbab4.xlsx
# OUTPUT : Hasil evaluasi + confusion matrix semua model
# ============================================================

import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")


# ============================================================
# 0. SETTING FILE
# ============================================================

BASE_DIR = Path("/Users/gitajilanmahira/Downloads/Gita Jilan_Rifaskes 2")
FILE_BERSIH = BASE_DIR / "Siapbab4.xlsx"

OUTPUT_DIR = BASE_DIR / "HASIL_MODEL_FINAL_FIXED"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PLOT_DIR = OUTPUT_DIR / "CONFUSION_MATRIX_PLOTS"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "Y_Final_PKM"

BEST_THRESHOLD = 0.37

BEST_PARAMS_RF = {
    "n_estimators": 1000,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "max_depth": None,
    "bootstrap": False,
    "random_state": 42,
    "n_jobs": -1
}

print("=" * 80)
print("SCRIPT MODEL FINAL FIXED PARAMETER")
print("=" * 80)
print("File input :", FILE_BERSIH)
print("Output dir :", OUTPUT_DIR)
print("Threshold  :", BEST_THRESHOLD)
print("Best params:")
print(BEST_PARAMS_RF)


# ============================================================
# 1. BACA DATA FINAL
# ============================================================

df_ml = pd.read_excel(FILE_BERSIH)
df_ml.columns = df_ml.columns.str.strip()

print("\n" + "=" * 80)
print("DATA FINAL")
print("=" * 80)
print("Shape df_ml:", df_ml.shape)

if TARGET_COL not in df_ml.columns:
    raise ValueError(f"Kolom target {TARGET_COL} tidak ditemukan.")

# ============================================================
# 2. CEK PENGAMAN
# ============================================================

VARIABEL_HAPUS_MODEL = [
    "B4R14",
    "B3R11E",
    "B9R13A",
    "B4R16A",
    "B3R13A"
]

print("\nCek 5 variabel yang harus sudah hilang:")
for col in VARIABEL_HAPUS_MODEL:
    print(f"{col} ada? {col in df_ml.columns}")

print("\nCek Dinkes:")
print("Total_Kelompok_Obat_Dinkes ada?", "Total_Kelompok_Obat_Dinkes" in df_ml.columns)
print("X_Ketersediaan_Dinkes ada?", "X_Ketersediaan_Dinkes" in df_ml.columns)


# ============================================================
# 3. BUAT X DAN y
# ============================================================

y = df_ml[TARGET_COL].copy()

daftar_hitam = [
    TARGET_COL,
    "Total_Kelompok_Obat_PKM",
    "ID_cases",
    "B1R1",
    "B1R2",
    "X_Ketersediaan_Dinkes"
]

for col in df_ml.columns:
    if col.startswith("Y_") or "B1281" in col or "B66" in col:
        if col not in daftar_hitam:
            daftar_hitam.append(col)

X = df_ml.drop(columns=[c for c in daftar_hitam if c in df_ml.columns]).copy()

print("\n" + "=" * 80)
print("X DAN y")
print("=" * 80)
print("Shape X:", X.shape)
print("Shape y:", y.shape)

print("\nDistribusi y seluruh data:")
print(y.value_counts().sort_index())
print((y.value_counts(normalize=True).sort_index() * 100).round(2))


# ============================================================
# 4. DEFINISI KOLOM MODEL
# ============================================================

kolom_passthrough = [
    "B3R3",
    "B1282R6",
    "B14R15BTERSEDIA",
    "B121R1",
    "B121R7",
    "B1282R1",
    "B1282R3",
    "B122R11",
    "B51R2DK2",
    "B3R22E",
    "B1282R5",
    "B3R40",
    "B126R4A",
    "B3R10",
    "B3R9B",
    "B14R2",
    "B8R2K4N",
    "B4R10B",
    "B121R5",
    "B121R10",
    "X_Persentase_Pemenuhan_EPurchasing",
    "X_Sumber_Dana_Umum",
    "X_Kondisi_Ruang_Farmasi",
    "X_Kualitas_Gudang",
    "X_Proporsi_Kapitasi",
    "B4R3",
    "B3R24",
    "B14R16",
    "X_Intensitas_Kapitasi"
]

kolom_nominal = [
    "B3R6",
    "X_Pengiriman_RKO",
    "B121R4",
    "B122R3",
    "X_Metode_Pembelian_Kapitasi",
    "X_Sumber_Dana_BPJS",
    "B122R1",
    "B51R1B",
    "X_Kerusakan_Obat",
    "B14R7A",
    "X_Kepemilikan_Kendaraan",
    "X_Sistem_Pencatatan"
]

kolom_numerik = [
    "B8R2K3N",
    "B1282R2",
    "B1282R4",
    "B14R23",
    "Total_Kelompok_Obat_Dinkes"
]

kolom_passthrough_ada = [c for c in kolom_passthrough if c in X.columns]
kolom_nominal_ada = [c for c in kolom_nominal if c in X.columns]
kolom_numerik_ada = [c for c in kolom_numerik if c in X.columns]

print("\n" + "=" * 80)
print("KOLOM MODEL")
print("=" * 80)
print("Kolom passthrough:", len(kolom_passthrough_ada), kolom_passthrough_ada)
print("Kolom nominal    :", len(kolom_nominal_ada), kolom_nominal_ada)
print("Kolom numerik    :", len(kolom_numerik_ada), kolom_numerik_ada)


# ============================================================
# 5. SPLIT DATA STRATIFY Y
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\n" + "=" * 80)
print("SPLIT DATA")
print("=" * 80)
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nDistribusi y_train:")
print(y_train.value_counts().sort_index())
print((y_train.value_counts(normalize=True).sort_index() * 100).round(2))

print("\nDistribusi y_test:")
print(y_test.value_counts().sort_index())
print((y_test.value_counts(normalize=True).sort_index() * 100).round(2))


# ============================================================
# 6. FUNGSI EVALUASI DAN CONFUSION MATRIX
# ============================================================

def evaluasi_model(nama_model, y_true, y_pred, y_proba):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    hasil = {
        "model": nama_model,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0,
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "auroc": float(roc_auc_score(y_true, y_proba)),
        "auprc": float(average_precision_score(y_true, y_proba)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }

    return hasil, cm


def simpan_confusion_matrix(nama_model, cm, output_dir):
    safe_name = (
        nama_model
        .replace(" ", "_")
        .replace("+", "plus")
        .replace("/", "_")
        .replace(".", "")
        .replace(":", "")
    )

    # Simpan Excel nilai CM
    df_cm = pd.DataFrame(
        cm,
        index=[
            "Aktual Y=0 (<80% obat)",
            "Aktual Y=1 (>=80% obat)"
        ],
        columns=[
            "Prediksi Y=0 (<80% obat)",
            "Prediksi Y=1 (>=80% obat)"
        ]
    )

    excel_path = output_dir / f"Confusion_Matrix_{safe_name}.xlsx"
    df_cm.to_excel(excel_path)

    # Simpan PNG visual CM
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[
            "Y=0\n<80% obat",
            "Y=1\n>=80% obat"
        ]
    )

    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, values_format="d", colorbar=True)

    plt.title(
        f"Confusion Matrix\n{nama_model}",
        fontsize=12,
        fontweight="bold"
    )
    plt.xlabel("Prediksi Model")
    plt.ylabel("Kelas Aktual")
    plt.tight_layout()

    png_path = output_dir / f"Confusion_Matrix_{safe_name}.png"
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.close()

    return excel_path, png_path


# ============================================================
# 7. PREPROCESSOR
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        ("jalur_nom", OneHotEncoder(handle_unknown="ignore"), kolom_nominal_ada),
        ("jalur_num", StandardScaler(), kolom_numerik_ada)
    ],
    remainder="passthrough"
)


# ============================================================
# 8. MODEL 1: RANDOM FOREST BASELINE SEDERHANA
# ============================================================

print("\n" + "=" * 80)
print("MODEL 1: RF BASELINE SEDERHANA")
print("=" * 80)

rf_baseline = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

rf_baseline.fit(X_train, y_train)

y_pred_base = rf_baseline.predict(X_test)
y_proba_base = rf_baseline.predict_proba(X_test)[:, 1]

hasil_base, cm_base = evaluasi_model(
    "RF Baseline",
    y_test,
    y_pred_base,
    y_proba_base
)

print(hasil_base)
print("Confusion matrix:")
print(cm_base)

simpan_confusion_matrix(
    "RF Baseline",
    cm_base,
    PLOT_DIR
)


# ============================================================
# 9. MODEL 2: RF BASELINE + ENCODING/SCALING
# ============================================================

print("\n" + "=" * 80)
print("MODEL 2: RF BASELINE + ENCODING/SCALING")
print("=" * 80)

rf_pipeline_baseline = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline_baseline.fit(X_train, y_train)

y_pred_pipe = rf_pipeline_baseline.predict(X_test)
y_proba_pipe = rf_pipeline_baseline.predict_proba(X_test)[:, 1]

hasil_pipe, cm_pipe = evaluasi_model(
    "RF Baseline + Encoding Scaling",
    y_test,
    y_pred_pipe,
    y_proba_pipe
)

print(hasil_pipe)
print("Confusion matrix:")
print(cm_pipe)

simpan_confusion_matrix(
    "RF Baseline + Encoding Scaling",
    cm_pipe,
    PLOT_DIR
)


# ============================================================
# 10. MODEL 3: RF TUNED PARAMETER FINAL THRESHOLD 0.50
# ============================================================

print("\n" + "=" * 80)
print("MODEL 3: RF TUNED FIXED PARAMETER - THRESHOLD 0.50")
print("=" * 80)

rf_tuned_fixed = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(**BEST_PARAMS_RF))
])

rf_tuned_fixed.fit(X_train, y_train)

y_proba_tuned = rf_tuned_fixed.predict_proba(X_test)[:, 1]

y_pred_tuned_default = (y_proba_tuned >= 0.50).astype(int)

hasil_tuned_default, cm_tuned_default = evaluasi_model(
    "RF Tuned Fixed Threshold 0.50",
    y_test,
    y_pred_tuned_default,
    y_proba_tuned
)

print(hasil_tuned_default)
print("Confusion matrix:")
print(cm_tuned_default)

simpan_confusion_matrix(
    "RF Tuned Fixed Threshold 0.50",
    cm_tuned_default,
    PLOT_DIR
)


# ============================================================
# 11. MODEL 4: RF TUNED PARAMETER FINAL THRESHOLD 0.37
# ============================================================

print("\n" + "=" * 80)
print("MODEL 4: RF TUNED FIXED PARAMETER - THRESHOLD OPTIMAL 0.37")
print("=" * 80)

y_pred_final = (y_proba_tuned >= BEST_THRESHOLD).astype(int)

hasil_final, cm_final = evaluasi_model(
    "RF Tuned Fixed Optimized Threshold 0.37",
    y_test,
    y_pred_final,
    y_proba_tuned
)

print(hasil_final)
print("Confusion matrix final:")
print(cm_final)

simpan_confusion_matrix(
    "RF Tuned Fixed Optimized Threshold 0.37",
    cm_final,
    PLOT_DIR
)


# ============================================================
# 12. TABEL PERBANDINGAN MODEL
# ============================================================

df_hasil = pd.DataFrame([
    hasil_base,
    hasil_pipe,
    hasil_tuned_default,
    hasil_final
])

kolom_tampil = [
    "model",
    "accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "mcc",
    "auroc",
    "auprc",
    "tn",
    "fp",
    "fn",
    "tp"
]

df_hasil = df_hasil[kolom_tampil]

print("\n" + "=" * 80)
print("TABEL PERBANDINGAN MODEL")
print("=" * 80)
print(df_hasil.round(4).to_string(index=False))

output_hasil_excel = OUTPUT_DIR / "Tabel_Perbandingan_Model_Fixed_Params.xlsx"
df_hasil.to_excel(output_hasil_excel, index=False)

output_hasil_csv = OUTPUT_DIR / "Tabel_Perbandingan_Model_Fixed_Params.csv"
df_hasil.to_csv(output_hasil_csv, index=False)

print("\nTabel perbandingan model disimpan:")
print(output_hasil_excel)
print(output_hasil_csv)


# ============================================================
# 13. SIMPAN SEMUA CONFUSION MATRIX DALAM SATU EXCEL
# ============================================================

with pd.ExcelWriter(OUTPUT_DIR / "Semua_Confusion_Matrix.xlsx") as writer:
    pd.DataFrame(
        cm_base,
        index=["Aktual Y=0", "Aktual Y=1"],
        columns=["Prediksi Y=0", "Prediksi Y=1"]
    ).to_excel(writer, sheet_name="RF Baseline")

    pd.DataFrame(
        cm_pipe,
        index=["Aktual Y=0", "Aktual Y=1"],
        columns=["Prediksi Y=0", "Prediksi Y=1"]
    ).to_excel(writer, sheet_name="RF Enc Scaling")

    pd.DataFrame(
        cm_tuned_default,
        index=["Aktual Y=0", "Aktual Y=1"],
        columns=["Prediksi Y=0", "Prediksi Y=1"]
    ).to_excel(writer, sheet_name="RF Tuned 050")

    pd.DataFrame(
        cm_final,
        index=["Aktual Y=0", "Aktual Y=1"],
        columns=["Prediksi Y=0", "Prediksi Y=1"]
    ).to_excel(writer, sheet_name="RF Tuned 037")

print("\nSemua confusion matrix disimpan di:")
print(OUTPUT_DIR / "Semua_Confusion_Matrix.xlsx")


# ============================================================
# 14. FEATURE IMPORTANCE MODEL FINAL
# ============================================================

feature_names = rf_tuned_fixed.named_steps["prep"].get_feature_names_out()
importance = rf_tuned_fixed.named_steps["model"].feature_importances_

df_feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": importance
}).sort_values("importance", ascending=False).reset_index(drop=True)

output_fi_excel = OUTPUT_DIR / "Feature_Importance_Model_Final_Fixed_Params.xlsx"
df_feature_importance.to_excel(output_fi_excel, index=False)

print("\n" + "=" * 80)
print("TOP 20 FEATURE IMPORTANCE MODEL FINAL")
print("=" * 80)
print(df_feature_importance.head(20).to_string(index=False))

print("\nFeature importance disimpan di:")
print(output_fi_excel)


# ============================================================
# 15. SIMPAN METRICS FINAL JSON
# ============================================================

metrics_json = {
    "data": {
        "file_input": str(FILE_BERSIH),
        "shape_df": list(df_ml.shape),
        "shape_X": list(X.shape),
        "shape_X_train": list(X_train.shape),
        "shape_X_test": list(X_test.shape),
        "y_train_distribution": {
            "0": int((y_train == 0).sum()),
            "1": int((y_train == 1).sum())
        },
        "y_test_distribution": {
            "0": int((y_test == 0).sum()),
            "1": int((y_test == 1).sum())
        }
    },
    "best_params_rf": BEST_PARAMS_RF,
    "best_threshold": BEST_THRESHOLD,
    "model_comparison": df_hasil.to_dict(orient="records"),
    "final_confusion_matrix": {
        "tn": int(hasil_final["tn"]),
        "fp": int(hasil_final["fp"]),
        "fn": int(hasil_final["fn"]),
        "tp": int(hasil_final["tp"])
    },
    "final_metrics": {
        "accuracy": float(hasil_final["accuracy"]),
        "precision": float(hasil_final["precision"]),
        "recall": float(hasil_final["recall"]),
        "specificity": float(hasil_final["specificity"]),
        "f1": float(hasil_final["f1"]),
        "mcc": float(hasil_final["mcc"]),
        "auroc": float(hasil_final["auroc"]),
        "auprc": float(hasil_final["auprc"])
    }
}

output_json = OUTPUT_DIR / "model_metrics_fixed_params.json"

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(metrics_json, f, indent=2, ensure_ascii=False)

print("\nMetrics JSON disimpan di:")
print(output_json)


# ============================================================
# 16. FINAL CHECK
# ============================================================

print("\n" + "=" * 80)
print("FINAL CHECK OUTPUT")
print("=" * 80)

file_outputs = [
    output_hasil_excel,
    output_hasil_csv,
    OUTPUT_DIR / "Semua_Confusion_Matrix.xlsx",
    output_fi_excel,
    output_json
]

for f in file_outputs:
    print(("ADA  " if f.exists() else "BELUM"), f)

print("\nFolder gambar confusion matrix:")
print(PLOT_DIR)

print("\nIsi folder confusion matrix:")
for f in sorted(PLOT_DIR.glob("*")):
    print("-", f.name)

print("\n" + "=" * 80)
print("SELESAI")
print("=" * 80)
print("Model final: RF Tuned Fixed Optimized Threshold 0.37")
print("Confusion matrix final:")
print(cm_final)
print("Metrik final:")
print(pd.Series(hasil_final).to_string())
# ============================================================
# 17. GAMBAR RANDOM FOREST + SHAP FINAL
# ============================================================

print("\n" + "=" * 80)
print("MEMBUAT GAMBAR RANDOM FOREST DAN SHAP")
print("=" * 80)

FIGURE_DIR = OUTPUT_DIR / "FINAL_FIGURES_RF_SHAP"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 17A. MAPPING NAMA VARIABEL AGAR TIDAK MUNCUL KODE
# ============================================================

feature_name_map = {
    "B3R3": "Status Rawat Inap Puskesmas",
    "B3R6": "Karakteristik Wilayah Puskesmas",
    "B3R9B": "Status Akreditasi Puskesmas",
    "B3R10": "Status Pengelolaan Keuangan Puskesmas",
    "B8R2K3N": "Jumlah SDM/Pedoman Pengelolaan Obat",
    "B8R2K4N": "Pedoman Pengelolaan Obat",
    "B14R2": "Bendahara Dana Kapitasi JKN",
    "B14R7A": "Penggunaan Dana Kapitasi",
    "B14R15BTERSEDIA": "Ketersediaan Apoteker",
    "B14R16": "Penerapan KBK di Puskesmas",
    "B14R23": "Jumlah Peserta PBI",
    "B121R1": "Perencanaan Kebutuhan Obat",
    "B121R4": "Metode Penyusunan RKO",
    "B121R5": "Sistem Pencatatan dan Pelaporan Obat",
    "B121R7": "Ketersediaan LPLPO",
    "B121R10": "Frekuensi LPLPO",
    "B122R1": "Sumber Pengadaan Obat",
    "B122R3": "Mekanisme Pembelian Obat Kapitasi",
    "B122R11": "Ketersediaan Gudang Obat",
    "B126R4A": "Monitoring Pengelolaan Obat",
    "B1282R1": "Penerimaan Obat dari Dinkes",
    "B1282R2": "Frekuensi Penerimaan Obat dari Dinkes",
    "B1282R3": "Kecukupan Obat dari Dinkes",
    "B1282R4": "Jumlah/Frekuensi Obat Diterima dari Dinkes",
    "B1282R5": "Sumber Obat dari Dinkes",
    "B1282R6": "Penanggung Jawab Ketersediaan Obat",
    "B3R22E": "Akses Internet Puskesmas",
    "B3R24": "Ketersediaan Transportasi Umum",
    "B3R40": "Ketersediaan Internet",
    "B4R3": "Kapasitas Pelayanan Puskesmas",
    "B4R10B": "Pelayanan Rawat Darurat",
    "B51R1B": "Pendidikan Kepala Puskesmas",
    "B51R2DK2": "Penanggung Jawab UKP/Kefarmasian/Laboratorium",
    "X_Pengiriman_RKO": "Pengiriman RKO",
    "X_Metode_Pembelian_Kapitasi": "Metode Pembelian Obat Kapitasi",
    "X_Persentase_Pemenuhan_EPurchasing": "Pemenuhan Obat melalui E-Purchasing",
    "X_Sumber_Dana_Umum": "Sumber Dana Umum",
    "X_Kondisi_Ruang_Farmasi": "Kondisi Ruang Farmasi",
    "X_Kualitas_Gudang": "Kualitas Gudang Obat",
    "X_Sumber_Dana_BPJS": "Sumber Dana BPJS",
    "X_Proporsi_Kapitasi": "Proporsi Dana Kapitasi untuk Obat dan Operasional",
    "X_Sistem_Pencatatan": "Sistem Pencatatan Obat",
    "X_Kerusakan_Obat": "Kerusakan/Kedaluwarsa Obat",
    "X_Kepemilikan_Kendaraan": "Kepemilikan Kendaraan Operasional",
    "X_Intensitas_Kapitasi": "Intensitas Penggunaan Dana Kapitasi",
    "Total_Kelompok_Obat_Dinkes": "Total Kelompok Obat Tersedia di Dinkes"
}

feature_value_map = {
    "B3R6": {
        "0": "Tidak Ada SK",
        "1": "Terpencil/Sangat Terpencil",
        "2": "Perdesaan",
        "3": "Perkotaan",
        "0.0": "Tidak Ada SK",
        "1.0": "Terpencil/Sangat Terpencil",
        "2.0": "Perdesaan",
        "3.0": "Perkotaan"
    },
    "B3R9B": {
        "0": "Tidak Ada Akreditasi",
        "1": "Dasar",
        "2": "Madya",
        "3": "Utama",
        "4": "Paripurna",
        "0.0": "Tidak Ada Akreditasi",
        "1.0": "Dasar",
        "2.0": "Madya",
        "3.0": "Utama",
        "4.0": "Paripurna"
    }
}

def clean_feature_name(x):
    x = str(x)
    x = x.replace("jalur_num__", "")
    x = x.replace("jalur_nom__", "")
    x = x.replace("remainder__", "")
    return x

def readable_name(x):
    cleaned = clean_feature_name(x)

    if cleaned in feature_name_map:
        return feature_name_map[cleaned]

    for kode, nama in sorted(feature_name_map.items(), key=lambda z: len(z[0]), reverse=True):
        if cleaned.startswith(kode + "_"):
            kategori = cleaned.replace(kode + "_", "")
            kategori_label = feature_value_map.get(kode, {}).get(kategori, kategori)
            return f"{nama}: {kategori_label}"

    return cleaned


# ============================================================
# 17B. RANDOM FOREST FEATURE IMPORTANCE BAR PLOT
# ============================================================

feature_names_code = rf_tuned_fixed.named_steps["prep"].get_feature_names_out()
feature_names_readable = [readable_name(f) for f in feature_names_code]
importance = rf_tuned_fixed.named_steps["model"].feature_importances_

df_feature_importance_readable = pd.DataFrame({
    "feature_code": feature_names_code,
    "feature_readable": feature_names_readable,
    "importance": importance
}).sort_values("importance", ascending=False).reset_index(drop=True)

df_feature_importance_readable.to_excel(
    FIGURE_DIR / "RF_Feature_Importance_Readable.xlsx",
    index=False
)

top20 = df_feature_importance_readable.head(20).iloc[::-1]

plt.figure(figsize=(10, 8))
plt.barh(top20["feature_readable"], top20["importance"])
plt.title("Top 20 Feature Importance - Random Forest Tuned", fontsize=14, fontweight="bold")
plt.xlabel("Importance")
plt.ylabel("Variabel")
plt.tight_layout()

output_fi_png = FIGURE_DIR / "RF_Feature_Importance_Top20_Readable.png"
plt.savefig(output_fi_png, dpi=300, bbox_inches="tight")
plt.close()

print("✅ Gambar feature importance dibuat:")
print(output_fi_png)


# ============================================================
# 17C. GAMBAR CONTOH TREE DARI RANDOM FOREST
# ============================================================

from sklearn.tree import plot_tree

rf_model_only = rf_tuned_fixed.named_steps["model"]

plt.figure(figsize=(28, 14))
plot_tree(
    rf_model_only.estimators_[0],
    feature_names=feature_names_readable,
    class_names=[
        "Y=0 <80% obat",
        "Y=1 ≥80% obat"
    ],
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8
)

plt.title("Contoh Pohon Keputusan dalam Random Forest Final", fontsize=16, fontweight="bold")
plt.tight_layout()

output_tree_png = FIGURE_DIR / "RF_Example_Tree_Depth3.png"
plt.savefig(output_tree_png, dpi=300, bbox_inches="tight")
plt.close()

print("✅ Gambar contoh tree Random Forest dibuat:")
print(output_tree_png)


# ============================================================
# 17D. SHAP BAR DAN BEESWARM
# ============================================================

try:
    import shap
    from scipy import sparse

    print("\nMembuat SHAP...")

    X_test_transformed = rf_tuned_fixed.named_steps["prep"].transform(X_test)

    if sparse.issparse(X_test_transformed):
        X_test_transformed = X_test_transformed.toarray()

    X_test_shap = pd.DataFrame(
        X_test_transformed,
        columns=feature_names_readable,
        index=X_test.index
    )

    explainer = shap.TreeExplainer(rf_model_only)
    shap_values = explainer.shap_values(X_test_shap)

    if isinstance(shap_values, list):
        shap_values_class1 = shap_values[1]
    elif len(np.array(shap_values).shape) == 3:
        shap_values_class1 = shap_values[:, :, 1]
    else:
        shap_values_class1 = shap_values

    df_shap_global = pd.DataFrame({
        "feature_readable": X_test_shap.columns,
        "mean_abs_shap": np.abs(shap_values_class1).mean(axis=0)
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

    df_shap_global["rank"] = range(1, len(df_shap_global) + 1)

    output_shap_csv = FIGURE_DIR / "shap_global_importance.csv"
    df_shap_global.to_csv(output_shap_csv, index=False)

    print("✅ Tabel SHAP global dibuat:")
    print(output_shap_csv)

    # SHAP bar
    shap.summary_plot(
        shap_values_class1,
        X_test_shap,
        plot_type="bar",
        max_display=20,
        show=False
    )

    plt.title("Global SHAP Feature Importance - Random Forest Tuned", fontsize=14, fontweight="bold")
    plt.tight_layout()

    output_shap_bar = FIGURE_DIR / "SHAP_Global_Bar_RF.png"
    plt.savefig(output_shap_bar, dpi=300, bbox_inches="tight")
    plt.close()

    print("✅ Gambar SHAP bar dibuat:")
    print(output_shap_bar)

    # SHAP beeswarm
    shap.summary_plot(
        shap_values_class1,
        X_test_shap,
        max_display=20,
        show=False
    )

    plt.title("Global SHAP Beeswarm - Random Forest Tuned", fontsize=14, fontweight="bold")
    plt.tight_layout()

    output_shap_beeswarm = FIGURE_DIR / "SHAP_Global_Beeswarm_RF.png"
    plt.savefig(output_shap_beeswarm, dpi=300, bbox_inches="tight")
    plt.close()

    print("✅ Gambar SHAP beeswarm dibuat:")
    print(output_shap_beeswarm)

    print("\nTop 20 SHAP global:")
    print(df_shap_global.head(20).to_string(index=False))

except Exception as e:
    print("⚠️ SHAP gagal dibuat.")
    print("Penyebab:", e)
    print("Kalau library belum ada, jalankan dulu di terminal:")
    print("pip install shap")


print("\n" + "=" * 80)
print("SELESAI MEMBUAT FIGURE FINAL")
print("=" * 80)
print("Folder figure:")
print(FIGURE_DIR)



In [ ]:
# ============================================================
# SCRIPT STANDALONE EXPORT 8 FILE FINAL
# Tidak perlu notebook
# Tidak tuning ulang
# Langsung pakai parameter final Random Forest
# ============================================================

import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")


# ============================================================
# 0. SETTING PATH
# ============================================================

BASE_DIR = Path("/Users/gitajilanmahira/Downloads/Gita Jilan_Rifaskes 2")
FILE_BERSIH = BASE_DIR / "Siapbab4.xlsx"

OUTPUT_DIR = BASE_DIR / "HASIL_MODEL_FINAL_FIXED"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_EXPORT_DIR = OUTPUT_DIR / "EXPORT_8_FILE_FINAL"
FINAL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "Y_Final_PKM"

BEST_THRESHOLD = 0.37

BEST_PARAMS_RF = {
    "n_estimators": 1000,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "max_depth": None,
    "bootstrap": False,
    "random_state": 42,
    "n_jobs": -1
}

print("=" * 80)
print("EXPORT 8 FILE FINAL - STANDALONE SCRIPT")
print("=" * 80)
print("File input :", FILE_BERSIH)
print("Output dir :", FINAL_EXPORT_DIR)


# ============================================================
# 1. BACA DATA FINAL
# ============================================================

df_ml = pd.read_excel(FILE_BERSIH)
df_ml.columns = df_ml.columns.str.strip()

print("\nShape df_ml:", df_ml.shape)

if TARGET_COL not in df_ml.columns:
    raise ValueError(f"Kolom target {TARGET_COL} tidak ditemukan di data.")


# ============================================================
# 2. CEK VARIABEL FINAL
# ============================================================

VARIABEL_HAPUS_MODEL = [
    "B4R14",
    "B3R11E",
    "B9R13A",
    "B4R16A",
    "B3R13A"
]

print("\nCek 5 variabel yang harus sudah hilang:")
for col in VARIABEL_HAPUS_MODEL:
    print(f"{col} ada? {col in df_ml.columns}")

print("\nCek Dinkes:")
print("Total_Kelompok_Obat_Dinkes ada?", "Total_Kelompok_Obat_Dinkes" in df_ml.columns)
print("X_Ketersediaan_Dinkes ada?", "X_Ketersediaan_Dinkes" in df_ml.columns)


# ============================================================
# 3. BUAT X DAN y
# ============================================================

y = df_ml[TARGET_COL].copy()

daftar_hitam = [
    TARGET_COL,
    "Total_Kelompok_Obat_PKM",
    "ID_cases",
    "B1R1",
    "B1R2",
    "X_Ketersediaan_Dinkes"
]

for col in df_ml.columns:
    if col.startswith("Y_") or "B1281" in col or "B66" in col:
        if col not in daftar_hitam:
            daftar_hitam.append(col)

X = df_ml.drop(columns=[c for c in daftar_hitam if c in df_ml.columns]).copy()

print("\nShape X:", X.shape)
print("Shape y:", y.shape)

print("\nDistribusi target seluruh data:")
print(y.value_counts().sort_index())
print((y.value_counts(normalize=True).sort_index() * 100).round(2))


# ============================================================
# 4. DEFINISI KOLOM MODEL
# ============================================================

kolom_passthrough = [
    "B3R3",
    "B1282R6",
    "B14R15BTERSEDIA",
    "B121R1",
    "B121R7",
    "B1282R1",
    "B1282R3",
    "B122R11",
    "B51R2DK2",
    "B3R22E",
    "B1282R5",
    "B3R40",
    "B126R4A",
    "B3R10",
    "B3R9B",
    "B14R2",
    "B8R2K4N",
    "B4R10B",
    "B121R5",
    "B121R10",
    "X_Persentase_Pemenuhan_EPurchasing",
    "X_Sumber_Dana_Umum",
    "X_Kondisi_Ruang_Farmasi",
    "X_Kualitas_Gudang",
    "X_Proporsi_Kapitasi",
    "B4R3",
    "B3R24",
    "B14R16",
    "X_Intensitas_Kapitasi"
]

kolom_nominal = [
    "B3R6",
    "X_Pengiriman_RKO",
    "B121R4",
    "B122R3",
    "X_Metode_Pembelian_Kapitasi",
    "X_Sumber_Dana_BPJS",
    "B122R1",
    "B51R1B",
    "X_Kerusakan_Obat",
    "B14R7A",
    "X_Kepemilikan_Kendaraan",
    "X_Sistem_Pencatatan"
]

kolom_numerik = [
    "B8R2K3N",
    "B1282R2",
    "B1282R4",
    "B14R23",
    "Total_Kelompok_Obat_Dinkes"
]

kolom_passthrough_ada = [c for c in kolom_passthrough if c in X.columns]
kolom_nominal_ada = [c for c in kolom_nominal if c in X.columns]
kolom_numerik_ada = [c for c in kolom_numerik if c in X.columns]

print("\nKolom passthrough:", len(kolom_passthrough_ada))
print("Kolom nominal    :", len(kolom_nominal_ada))
print("Kolom numerik    :", len(kolom_numerik_ada))


# ============================================================
# 5. SPLIT TRAIN TEST
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nX_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nDistribusi y_train:")
print(y_train.value_counts().sort_index())

print("\nDistribusi y_test:")
print(y_test.value_counts().sort_index())


# ============================================================
# 6. PREPROCESSOR DAN MODEL FINAL
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        ("jalur_nom", OneHotEncoder(handle_unknown="ignore"), kolom_nominal_ada),
        ("jalur_num", StandardScaler(), kolom_numerik_ada)
    ],
    remainder="passthrough"
)

rf_tuned_fixed = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(**BEST_PARAMS_RF))
])

print("\nTraining model final fixed parameter...")
rf_tuned_fixed.fit(X_train, y_train)

y_proba_final = rf_tuned_fixed.predict_proba(X_test)[:, 1]
y_pred_final = (y_proba_final >= BEST_THRESHOLD).astype(int)


# ============================================================
# 7. EVALUASI MODEL FINAL
# ============================================================

cm_final = confusion_matrix(y_test, y_pred_final, labels=[0, 1])
tn, fp, fn, tp = cm_final.ravel()

hasil_final = {
    "accuracy": float(accuracy_score(y_test, y_pred_final)),
    "precision": float(precision_score(y_test, y_pred_final, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred_final, zero_division=0)),
    "specificity": float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0,
    "f1": float(f1_score(y_test, y_pred_final, zero_division=0)),
    "mcc": float(matthews_corrcoef(y_test, y_pred_final)),
    "auroc": float(roc_auc_score(y_test, y_proba_final)),
    "auprc": float(average_precision_score(y_test, y_proba_final)),
    "tn": int(tn),
    "fp": int(fp),
    "fn": int(fn),
    "tp": int(tp)
}

print("\n" + "=" * 80)
print("HASIL FINAL")
print("=" * 80)
print(pd.Series(hasil_final))

print("\nConfusion matrix final:")
print(cm_final)


# ============================================================
# 8. MAPPING NAMA VARIABEL
# ============================================================

feature_dictionary_map = {
    "B3R3": "Status rawat inap Puskesmas",
    "B3R10": "Status pengelolaan keuangan Puskesmas",
    "B3R9B": "Status akreditasi Puskesmas",
    "B3R6": "Karakteristik wilayah Puskesmas",
    "B14R2": "Bendahara dana kapitasi JKN",
    "B14R23": "Jumlah peserta PBI",
    "B14R15BTERSEDIA": "Ketersediaan apoteker",
    "B1282R6": "Penanggung jawab ketersediaan obat",
    "B8R2K3N": "Jumlah SDM atau pedoman pengelolaan obat",
    "B8R2K4N": "Pedoman pengelolaan obat",
    "B4R10B": "Pelayanan rawat darurat",
    "B121R1": "Perencanaan kebutuhan obat",
    "B121R4": "Metode penyusunan RKO",
    "B121R5": "Sistem pencatatan dan pelaporan obat",
    "B121R7": "Ketersediaan LPLPO",
    "B121R10": "Frekuensi LPLPO",
    "B122R3": "Mekanisme pembelian obat kapitasi",
    "B1282R1": "Penerimaan obat dari Dinkes",
    "B1282R2": "Frekuensi penerimaan obat dari Dinkes",
    "B1282R3": "Kecukupan obat dari Dinkes",
    "B1282R4": "Jumlah atau frekuensi obat diterima dari Dinkes",
    "B1282R5": "Sumber obat dari Dinkes",
    "B122R11": "Ketersediaan gudang obat",
    "B122R1": "Sumber pengadaan obat",
    "B4R3": "Kapasitas pelayanan Puskesmas",
    "B51R1B": "Pendidikan kepala Puskesmas",
    "B3R22E": "Akses internet Puskesmas",
    "B3R24": "Ketersediaan transportasi umum",
    "B3R40": "Ketersediaan internet",
    "B51R2DK2": "Penanggung jawab UKP/kefarmasian/laboratorium",
    "B14R16": "Penerapan KBK di Puskesmas",
    "B14R7A": "Penggunaan dana kapitasi",
    "B126R4A": "Monitoring pengelolaan obat",
    "X_Pengiriman_RKO": "Pengiriman RKO",
    "X_Metode_Pembelian_Kapitasi": "Metode pembelian obat kapitasi",
    "X_Persentase_Pemenuhan_EPurchasing": "Persentase pemenuhan e-purchasing",
    "X_Sumber_Dana_Umum": "Sumber dana umum",
    "X_Kondisi_Ruang_Farmasi": "Kondisi ruang farmasi",
    "X_Kualitas_Gudang": "Kualitas gudang obat",
    "X_Sumber_Dana_BPJS": "Sumber dana BPJS",
    "X_Proporsi_Kapitasi": "Proporsi dana kapitasi",
    "X_Sistem_Pencatatan": "Sistem pencatatan obat",
    "X_Kerusakan_Obat": "Kerusakan atau kedaluwarsa obat",
    "X_Kepemilikan_Kendaraan": "Kepemilikan kendaraan operasional",
    "X_Intensitas_Kapitasi": "Intensitas penggunaan dana kapitasi",
    "Total_Kelompok_Obat_Dinkes": "Total kelompok obat tersedia di Dinas Kesehatan"
}

def clean_feature_name(x):
    x = str(x)
    x = x.replace("jalur_num__", "")
    x = x.replace("jalur_nom__", "")
    x = x.replace("remainder__", "")
    return x

def readable_name(x):
    cleaned = clean_feature_name(x)

    if cleaned in feature_dictionary_map:
        return feature_dictionary_map[cleaned]

    for kode, nama in sorted(feature_dictionary_map.items(), key=lambda z: len(z[0]), reverse=True):
        if cleaned.startswith(kode + "_"):
            kategori = cleaned.replace(kode + "_", "")
            return f"{nama}: {kategori}"

    return cleaned


# ============================================================
# 9. EXPORT 8 FILE FINAL
# ============================================================

print("\n" + "=" * 80)
print("EXPORT 8 FILE FINAL")
print("=" * 80)

# ------------------------------------------------------------
# 1. model_metrics.json
# ------------------------------------------------------------

model_metrics = {
    "judul_penelitian": "Pengembangan Model Klasifikasi Machine Learning untuk Prediksi Ketersediaan Obat Esensial di Puskesmas Indonesia",
    "model_final": "Random Forest Tuned + Optimized Threshold",
    "threshold_final": float(BEST_THRESHOLD),
    "best_params": {
        "n_estimators": 1000,
        "min_samples_split": 2,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
        "max_depth": None,
        "bootstrap": False,
        "random_state": 42
    },
    "data_shape": {
        "df_ml": list(df_ml.shape),
        "X": list(X.shape),
        "X_train": list(X_train.shape),
        "X_test": list(X_test.shape)
    },
    "target_distribution_all": {
        "0": int((y == 0).sum()),
        "1": int((y == 1).sum())
    },
    "target_distribution_train": {
        "0": int((y_train == 0).sum()),
        "1": int((y_train == 1).sum())
    },
    "target_distribution_test": {
        "0": int((y_test == 0).sum()),
        "1": int((y_test == 1).sum())
    },
    "final_metrics": {
        "accuracy": hasil_final["accuracy"],
        "precision": hasil_final["precision"],
        "recall": hasil_final["recall"],
        "specificity": hasil_final["specificity"],
        "f1": hasil_final["f1"],
        "mcc": hasil_final["mcc"],
        "auroc": hasil_final["auroc"],
        "auprc": hasil_final["auprc"]
    },
    "confusion_matrix_final": {
        "tn": hasil_final["tn"],
        "fp": hasil_final["fp"],
        "fn": hasil_final["fn"],
        "tp": hasil_final["tp"]
    }
}

file_1 = FINAL_EXPORT_DIR / "model_metrics.json"

with open(file_1, "w", encoding="utf-8") as f:
    json.dump(model_metrics, f, indent=4, ensure_ascii=False)

print("1. model_metrics.json selesai")


# ------------------------------------------------------------
# 2. feature_importance.csv
# ------------------------------------------------------------

feature_names = rf_tuned_fixed.named_steps["prep"].get_feature_names_out()
importance = rf_tuned_fixed.named_steps["model"].feature_importances_

df_feature_importance_final = pd.DataFrame({
    "feature": feature_names,
    "feature_readable": [readable_name(f) for f in feature_names],
    "importance": importance
}).sort_values("importance", ascending=False).reset_index(drop=True)

file_2 = FINAL_EXPORT_DIR / "feature_importance.csv"
df_feature_importance_final.to_csv(file_2, index=False, encoding="utf-8-sig")

print("2. feature_importance.csv selesai")


# ------------------------------------------------------------
# 3. feature_dictionary.csv
# ------------------------------------------------------------

df_feature_dictionary = pd.DataFrame({
    "nama_variabel": X.columns,
    "deskripsi_variabel": [
        feature_dictionary_map.get(col, "Deskripsi belum tersedia")
        for col in X.columns
    ],
    "jenis_penggunaan": [
        "Numerik" if col in kolom_numerik_ada else
        "Nominal/One-hot encoding" if col in kolom_nominal_ada else
        "Ordinal/Biner/Passthrough"
        for col in X.columns
    ],
    "digunakan_dalam_model": "Ya"
})

file_3 = FINAL_EXPORT_DIR / "feature_dictionary.csv"
df_feature_dictionary.to_csv(file_3, index=False, encoding="utf-8-sig")

print("3. feature_dictionary.csv selesai")


# ------------------------------------------------------------
# 4. y_variable_definition.txt
# ------------------------------------------------------------

text_y_definition = f"""
DEFINISI VARIABEL TARGET

Nama variabel target:
Y_Final_PKM

Definisi:
Y_Final_PKM merupakan variabel target biner yang menunjukkan status ketersediaan obat esensial di Puskesmas.

Kriteria pembentukan:
- Setiap Puskesmas dinilai berdasarkan total kelompok obat esensial yang tersedia.
- Total kelompok obat esensial yang digunakan adalah 40 kelompok obat.
- Ambang batas ketersediaan obat esensial ditetapkan sebesar 80%.
- 80% dari 40 kelompok obat = 32 kelompok obat.

Kategori:
- Y_Final_PKM = 1 apabila Puskesmas memiliki minimal 32 kelompok obat esensial tersedia atau setara dengan ≥80%.
- Y_Final_PKM = 0 apabila Puskesmas memiliki kurang dari 32 kelompok obat esensial tersedia atau setara dengan <80%.

Distribusi target pada data final:
- Y=0: {int((y == 0).sum())} Puskesmas atau {(y.value_counts(normalize=True).sort_index()[0] * 100):.2f}%
- Y=1: {int((y == 1).sum())} Puskesmas atau {(y.value_counts(normalize=True).sort_index()[1] * 100):.2f}%

Model final:
Random Forest Tuned dengan threshold optimal {BEST_THRESHOLD}.
"""

file_4 = FINAL_EXPORT_DIR / "y_variable_definition.txt"

with open(file_4, "w", encoding="utf-8") as f:
    f.write(text_y_definition.strip())

print("4. y_variable_definition.txt selesai")


# ------------------------------------------------------------
# 5. X_train.csv
# ------------------------------------------------------------

file_5 = FINAL_EXPORT_DIR / "X_train.csv"
X_train.reset_index(drop=True).to_csv(file_5, index=False, encoding="utf-8-sig")

print("5. X_train.csv selesai")


# ------------------------------------------------------------
# 6. X_test.csv
# ------------------------------------------------------------

file_6 = FINAL_EXPORT_DIR / "X_test.csv"
X_test.reset_index(drop=True).to_csv(file_6, index=False, encoding="utf-8-sig")

print("6. X_test.csv selesai")


# ------------------------------------------------------------
# 7. y_train.csv
# ------------------------------------------------------------

file_7 = FINAL_EXPORT_DIR / "y_train.csv"
pd.DataFrame({"Y_Final_PKM": y_train.reset_index(drop=True)}).to_csv(
    file_7,
    index=False,
    encoding="utf-8-sig"
)

print("7. y_train.csv selesai")


# ------------------------------------------------------------
# 8. y_test.csv
# ------------------------------------------------------------

file_8 = FINAL_EXPORT_DIR / "y_test.csv"
pd.DataFrame({"Y_Final_PKM": y_test.reset_index(drop=True)}).to_csv(
    file_8,
    index=False,
    encoding="utf-8-sig"
)

print("8. y_test.csv selesai")


# ============================================================
# 10. BONUS FILE
# ============================================================

file_bonus_model = FINAL_EXPORT_DIR / "model_final_random_forest.pkl"
joblib.dump(rf_tuned_fixed, file_bonus_model)

df_cm_final_export = pd.DataFrame(
    cm_final,
    index=["Aktual Y=0 (<80% obat)", "Aktual Y=1 (≥80% obat)"],
    columns=["Prediksi Y=0 (<80% obat)", "Prediksi Y=1 (≥80% obat)"]
)

file_bonus_cm = FINAL_EXPORT_DIR / "confusion_matrix_final.csv"
df_cm_final_export.to_csv(file_bonus_cm, encoding="utf-8-sig")

text_summary = f"""
RINGKASAN MODEL FINAL

Model final:
Random Forest Tuned + Optimized Threshold

Parameter model:
- n_estimators: 1000
- min_samples_split: 2
- min_samples_leaf: 2
- max_features: sqrt
- max_depth: None
- bootstrap: False

Threshold final:
{BEST_THRESHOLD}

Jumlah data:
- Total data final: {df_ml.shape[0]}
- Jumlah fitur model: {X.shape[1]}
- Data latih: {X_train.shape[0]}
- Data uji: {X_test.shape[0]}

Distribusi target:
- Y=0 seluruh data: {int((y == 0).sum())}
- Y=1 seluruh data: {int((y == 1).sum())}

Metrik final:
- Accuracy: {hasil_final["accuracy"]:.4f}
- Precision: {hasil_final["precision"]:.4f}
- Recall: {hasil_final["recall"]:.4f}
- Specificity: {hasil_final["specificity"]:.4f}
- F1 Score: {hasil_final["f1"]:.4f}
- MCC: {hasil_final["mcc"]:.4f}
- AUROC: {hasil_final["auroc"]:.4f}
- AUPRC: {hasil_final["auprc"]:.4f}

Confusion matrix final:
- TN: {hasil_final["tn"]}
- FP: {hasil_final["fp"]}
- FN: {hasil_final["fn"]}
- TP: {hasil_final["tp"]}
"""

file_bonus_summary = FINAL_EXPORT_DIR / "model_development_summary.txt"

with open(file_bonus_summary, "w", encoding="utf-8") as f:
    f.write(text_summary.strip())

print("\nBonus file selesai:")
print("-", file_bonus_model.name)
print("-", file_bonus_cm.name)
print("-", file_bonus_summary.name)


# ============================================================
# 11. FINAL CHECK
# ============================================================

print("\n" + "=" * 80)
print("CEK 8 FILE FINAL")
print("=" * 80)

final_files = [
    file_1,
    file_2,
    file_3,
    file_4,
    file_5,
    file_6,
    file_7,
    file_8
]

for i, file in enumerate(final_files, start=1):
    status = "ADA" if file.exists() else "BELUM ADA"
    print(f"{i}. {status} - {file.name}")

print("\nFile bonus:")
bonus_files = [
    file_bonus_model,
    file_bonus_cm,
    file_bonus_summary
]

for file in bonus_files:
    status = "ADA" if file.exists() else "BELUM ADA"
    print(f"- {status} - {file.name}")

print("\nFolder export final:")
print(FINAL_EXPORT_DIR)

print("\n" + "=" * 80)
print("SELESAI. 8 FILE FINAL SUDAH DIEKSPOR.")
print("=" * 80)